# Synthetic Data Generation Tutorial

This tutorial demonstrates how to use SDG Hub to generate synthetic question-answer pairs from documents using various large language models. We'll cover:

1. Setting up the environment
2. Configuring different model backends (OpenAI, vLLM, Ollama, etc.)
3. Configuring the data generation pipeline
4. Generating synthetic data
5. Analyzing results

**📋 See [Model Serving Guide](../../../docs/model_serving.md) for detailed backend configuration.**

# Synthetic Data Generation Tutorial

This tutorial demonstrates how to use SDG Hub to generate synthetic question-answer pairs from documents using various large language models. We'll cover:

1. Setting up the environment
2. Configuring different model backends (OpenAI, vLLM, Ollama, etc.)
3. Configuring the data generation pipeline
4. Generating synthetic data
5. Analyzing results

**📋 See [MODEL_BACKENDS.md](MODEL_BACKENDS.md) for detailed backend configuration guide.**

In [1]:
# Enable auto-reloading of modules - useful during development
%load_ext autoreload
%autoreload 2

### Setup Instructions

**Option 1: Install from PyPI**
```bash 
pip install sdg-hub
```

**Option 2: Install from source (for development)**
```bash
git clone https://github.com/Red-Hat-AI-Innovation-Team/sdg_hub.git
cd sdg_hub
pip install -e .
```

**Option 3: Using UV (recommended)**
```bash
uv sync --extra dev
```

In [ ]:
### Configuration Variables

Configure your model backend here. See the [Model Serving Guide](../../../docs/model_serving.md) for detailed setup instructions.

```python
# Configuration options - choose one backend

# Option 1: OpenAI (Cloud)
BACKEND = "openai"
API_KEY = "your-openai-api-key"  # Or set OPENAI_API_KEY env var
BASE_URL = "https://api.openai.com/v1"
MODEL_ID = "gpt-4"

# Option 2: Local vLLM server
# BACKEND = "vllm"
# API_KEY = "EMPTY"
# BASE_URL = "http://localhost:8000/v1"
# MODEL_ID = "meta-llama/Llama-3.3-70B-Instruct"

# Option 3: Ollama
# BACKEND = "ollama"
# API_KEY = "EMPTY"
# BASE_URL = "http://localhost:11434/v1"
# MODEL_ID = "llama3.1"

# Option 4: Azure OpenAI
# BACKEND = "azure"
# API_KEY = "your-azure-api-key"
# BASE_URL = "https://your-resource.openai.azure.com/"
# MODEL_ID = "your-deployment-name"
```

### Configuration Variables

Configure your model backend here. See [MODEL_BACKENDS.md](MODEL_BACKENDS.md) for detailed setup instructions.

```python
# Configuration options - choose one backend

# Option 1: OpenAI (Cloud)
BACKEND = "openai"
API_KEY = "your-openai-api-key"  # Or set OPENAI_API_KEY env var
BASE_URL = "https://api.openai.com/v1"
MODEL_ID = "gpt-4"

# Option 2: Local vLLM server
# BACKEND = "vllm"
# API_KEY = "EMPTY"
# BASE_URL = "http://localhost:8000/v1"
# MODEL_ID = "meta-llama/Llama-3.3-70B-Instruct"

# Option 3: Ollama
# BACKEND = "ollama"
# API_KEY = "EMPTY"
# BASE_URL = "http://localhost:11434/v1"
# MODEL_ID = "llama3.1"

# Option 4: Azure OpenAI
# BACKEND = "azure"
# API_KEY = "your-azure-api-key"
# BASE_URL = "https://your-resource.openai.azure.com/"
# MODEL_ID = "your-deployment-name"
```

In [ ]:
# Backend Configuration
# Uncomment and modify the backend you want to use

# Option 1: OpenAI (default for this example)
BACKEND = "openai"
API_KEY = os.getenv("OPENAI_API_KEY", "your-openai-api-key")
BASE_URL = "https://api.openai.com/v1"
MODEL_ID = "gpt-4"

# Option 2: Local vLLM server (uncomment to use)
# BACKEND = "vllm"
# API_KEY = "EMPTY"
# BASE_URL = "http://localhost:8000/v1"
# MODEL_ID = "meta-llama/Llama-3.3-70B-Instruct"

# Option 3: Ollama (uncomment to use)
# BACKEND = "ollama" 
# API_KEY = "EMPTY"
# BASE_URL = "http://localhost:11434/v1"
# MODEL_ID = "llama3.1"

# Initialize client
client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
)

# Verify connection and get available models
try:
    models = client.models.list()
    print(f"✅ Connected to {BACKEND} backend")
    print("Available models:")
    for model in models.data[:5]:  # Show first 5 models
        print(f"  - {model.id}")
    
    # Use specified model or first available
    if MODEL_ID in [m.id for m in models.data]:
        teacher_model = MODEL_ID
        print(f"🎯 Using specified model: {teacher_model}")
    else:
        teacher_model = models.data[0].id
        print(f"⚠️  Model '{MODEL_ID}' not found, using: {teacher_model}")
        
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("Please check your configuration and ensure the server is running.")

### Configure the Data Generation Pipeline

We'll create a pipeline configuration that adapts to your model's capabilities.

In [ ]:
# Choose appropriate YAML config based on your model
CONFIG_MAP = {
    "gpt-4": "synth_knowledge1.5_llama3.3.yaml",
    "gpt-3.5-turbo": "synth_knowledge1.5_llama3.3.yaml", 
    "meta-llama/Llama-3.3-70B-Instruct": "synth_knowledge1.5_llama3.3.yaml",
    "llama3.1": "synth_knowledge1.5_llama3.3.yaml",
    # Add more model mappings as needed
}

# Get config file for the model (default to llama3.3 config)
config_file = CONFIG_MAP.get(teacher_model, "synth_knowledge1.5_llama3.3.yaml")
print(f"Using configuration: {config_file}")

# Load the flow configuration
try:
    flow_cfg = Flow(client).get_flow_from_file(config_file)
    print(f"✅ Flow configuration loaded successfully")
except Exception as e:
    print(f"❌ Failed to load flow configuration: {e}")
    print("Make sure the YAML config file exists in the current directory")

# Initialize the SDG pipeline
sdg = SDG(
    [flow_cfg],         # Use Flow directly (not wrapped in Pipeline)
    num_workers=1,      # Number of parallel workers
    batch_size=1,       # Batch size for processing  
    save_freq=1000,     # How often to save checkpoints
)

### Load and Prepare Seed Data

We'll load our seed data (documents) that will be used to generate question-answer pairs.

In [ ]:
# Option 1: Load from JSON file (replace with your data path)
seed_data_path = os.getenv("SEED_DATA_PATH", "your_data.json")

if seed_data_path != "your_data.json" and os.path.exists(seed_data_path):
    print(f"Loading data from: {seed_data_path}")
    ds = load_dataset('json', data_files=seed_data_path, split='train')
    print(f"Loaded {len(ds)} documents")
else:
    # Option 2: Use sample data for demonstration
    print("Using sample data for demonstration")
    sample_documents = [
        {
            "document": """
            Artificial Intelligence (AI) is a branch of computer science that aims to create intelligent machines 
            that can perform tasks that typically require human intelligence. These tasks include learning, reasoning, 
            problem-solving, perception, and language understanding. AI systems can be categorized into two main types: 
            narrow AI, which is designed for specific tasks, and general AI, which would have human-like cognitive abilities 
            across multiple domains. Machine learning, a subset of AI, enables computers to learn and improve from experience 
            without being explicitly programmed for every task.
            """
        },
        {
            "document": """
            Climate change refers to long-term shifts in global temperatures and weather patterns. While climate variations 
            are natural, scientific evidence shows that human activities have been the primary driver of climate change since 
            the mid-20th century. The burning of fossil fuels releases greenhouse gases like carbon dioxide into the atmosphere, 
            which trap heat and warm the planet. This warming leads to rising sea levels, more frequent extreme weather events, 
            changes in precipitation patterns, and impacts on ecosystems and biodiversity.
            """
        }
    ]
    ds = Dataset.from_list(sample_documents)
    print(f"Created sample dataset with {len(ds)} documents")

# For testing, use a subset
test_size = min(2, len(ds))
ds = ds.select(range(test_size))
print(f"Using {len(ds)} document(s) for generation")

# Preview the data
print("\\nSample document preview:")
print(ds[0]['document'][:200] + "..." if len(ds[0]['document']) > 200 else ds[0]['document'])

### Generate Synthetic Data

Now we'll generate synthetic question-answer pairs from the documents.

In [ ]:
# Generate synthetic data
checkpoint_dir = os.getenv("CHECKPOINT_DIR", "checkpoints")
print(f"Starting data generation with {BACKEND} backend...")
print(f"Checkpoint directory: {checkpoint_dir}")

try:
    generated_data = sdg.generate(ds, checkpoint_dir=checkpoint_dir)
    print(f"✅ Successfully generated {len(generated_data)} examples")
    
    # Display sample results
    if len(generated_data) > 0:
        print("\\n🎯 Sample Generated Data:")
        print("="*50)
        sample = generated_data[0]
        
        for key, value in sample.items():
            if key in ['question', 'response', 'document']:
                print(f"**{key.upper()}:**")
                display_text = str(value)[:300] + "..." if len(str(value)) > 300 else str(value)
                print(display_text)
                print("-" * 30)
                
except Exception as e:
    print(f"❌ Generation failed: {e}")
    import traceback
    traceback.print_exc()
    print("\\n💡 Troubleshooting tips:")
    print("- Check your model configuration")
    print("- Ensure sufficient context length for your model")
    print("- Verify your API key and endpoint")
    print("- Consider using a smaller document or chunking")

### Save and Analyze Results

Let's save the generated data and perform some basic analysis.

In [ ]:
# Save generated data
if 'generated_data' in locals() and generated_data is not None:
    output_dir = os.getenv("OUTPUT_DIR", "output")
    os.makedirs(output_dir, exist_ok=True)
    
    # Save as JSONL (recommended for datasets)
    jsonl_file = os.path.join(output_dir, f"generated_data_{BACKEND}.jsonl")
    generated_data.to_json(jsonl_file)
    print(f"💾 Saved data to: {jsonl_file}")
    
    # Save as CSV for easy viewing
    csv_file = os.path.join(output_dir, f"generated_data_{BACKEND}.csv")
    generated_data.to_csv(csv_file)
    print(f"💾 Saved data to: {csv_file}")
    
    # Basic analysis
    print(f"\\n📊 Dataset Analysis:")
    print(f"Total examples: {len(generated_data)}")
    print(f"Columns: {generated_data.column_names}")
    
    if 'question' in generated_data.column_names:
        avg_q_length = sum(len(q) for q in generated_data['question']) / len(generated_data)
        print(f"Average question length: {avg_q_length:.1f} characters")
        
    if 'response' in generated_data.column_names:
        avg_r_length = sum(len(r) for r in generated_data['response']) / len(generated_data)
        print(f"Average response length: {avg_r_length:.1f} characters")
        
else:
    print("❌ No data to save. Please run the generation step successfully first.")

### Next Steps and Customization

This notebook can be customized for your specific use case:

In [ ]:
print("🛠️ Customization Options:")
print("="*40)
print()

print("1. **Custom Data Format:**")
print("   Your JSON data should have 'document' field:")
print('   {"document": "Your text content here"}')
print()

print("2. **Model-Specific Configurations:**")
print("   - For small models (< 8B): Use chunking and reduce max_tokens")
print("   - For large models (70B+): Can use longer contexts")
print("   - See docs/model_serving.md for detailed setup")
print()

print("3. **Quality Control:**")
print("   The pipeline includes filtering steps for:")
print("   - Faithfulness (answers match document content)")
print("   - Relevancy (questions are relevant to content)")
print("   - Question quality (well-formed questions)")
print()

print("4. **Production Usage:**")
print("   For large datasets, consider:")
print("   - Increasing num_workers for parallel processing")
print("   - Using larger batch_size for efficiency")
print("   - Setting up proper checkpointing")
print("   - Monitoring token usage and costs")
print()

print("5. **Troubleshooting:**")
print("   Common issues and solutions:")
print("   - Context length errors → Use chunking or reduce max_tokens")
print("   - API rate limits → Reduce batch size or add delays")
print("   - Memory issues → Reduce num_workers or batch_size")
print("   - Quality issues → Adjust filtering thresholds")
print("   - See docs/model_serving.md for comprehensive troubleshooting")

### Resources and Further Reading

- **[Model Serving Guide](../../../docs/model_serving.md)** - Comprehensive guide for configuring different model backends
- **[SDG Hub Documentation](https://github.com/Red-Hat-AI-Innovation-Team/sdg_hub)** - Official repository and documentation
- **[Examples Directory](../../../examples/)** - Additional examples and use cases

**For production usage:**
```bash
# Command-line interface for large-scale generation
python -m sdg_hub.cli generate \
    --config-path your_config.yaml \
    --data-path your_data.jsonl \
    --output-path output_data.jsonl \
    --num-workers 4 \
    --batch-size 10
```

**Community and Support:**
- Open issues on [GitHub](https://github.com/Red-Hat-AI-Innovation-Team/sdg_hub/issues)
- Check existing discussions and examples
- Contribute improvements and new backends

---
🎉 **You've successfully completed the SDG Hub tutorial!** 

Your generated synthetic data is ready for use in training, evaluation, or further processing.

### Compare Generated Data

Let's compare the outputs from both models by saving them to a markdown file for easy review.

### Production Usage

For large-scale data generation, use the command-line script instead of this notebook:

```bash
python scripts/generate.py --ds_path seed_data.jsonl \
    --bs 2 --num_workers 10 \
    --save_path <your_save_path> \
    --flow ../src/sdg_hub/flows/generation/knowledge/synth_knowledge1.5.yaml \
    --checkpoint_dir <your_checkpoint_dir> \
    --endpoint <your_endpoint>
```

Note: For LLaMA 3.3, use `synth_knowledge1.5_llama3.3.yaml` as the flow configuration file.